In [148]:
import pandas as pd
import json
import numpy as np
import shutil
import os

In [69]:
dfs = []
for file in ['report_file_store_colab.csv', 'report_file_store.csv', 'report_file_store_kaggle.csv']:
    df = pd.read_csv(file)
    df['source'] = file.removeprefix('report_file_store').removeprefix('_').removesuffix('.csv') or 'local'
    dfs.append(df)

In [70]:
report = pd.concat(dfs, ignore_index=True)

In [71]:
report = report[report['error'].isna() & ~report['result'].isna()]

In [72]:
report['result'] = [eval(x, {'np': np}) for x in report['result']]

In [73]:
report['trial_start'] = [json.loads(x)[0] for x in report['step_start']]
report['trial_start'] = pd.to_datetime(report['trial_start'])

In [74]:
report['score_std'] = [np.std([x[0] for x in json.loads(y)['dt_score']]) for y in report['trace']]

In [75]:
report.columns

Index(['uuid', 'optuna_trial_number', 'tags', 'step_names', 'step_start',
       'step_stop', 'result', 'error', 'prune_reason', 'vars', 'trace',
       'source', 'trial_start', 'score_std'],
      dtype='str')

In [76]:
report = report[(report['trial_start'] > pd.to_datetime('2026-10-12'))]

In [169]:
arch = 'gru'
ds = 'FD004'
r = report[
    ~report['tags'].str.contains('supcon')
    & report['tags'].str.contains(arch)
    & ~report['tags'].str.contains('margin_absolute')
    & report['tags'].str.contains(ds)].sort_values('result').reset_index()
x = r.loc[0, ['result', 'score_std', 'source', 'uuid']]
#r['vars'].apply(print)
d = {'colab': 'filestore_colab', 'kaggle': 'results-kaggle-9/filestore', 'local': 'adbiasminfilestore'}[x['source']]
x, d
file = '/mnt/sghdd/' + d + '/' + str(x['uuid']) + '.pth'
dest = '/home/sasha/Documents/analise-db-iasmin-2026/models/' + arch + '-e2e-' + ds.lower() + '.pth'
assert not os.path.exists(dest)
shutil.copy(file, dest)

'/home/sasha/Documents/analise-db-iasmin-2026/models/gru-e2e-fd004.pth'